In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import librosa
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.string_utils import *

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/"
DESTINATION = "/workspaces/dev/output/LibriSpeechASRcorpus/sclient/whisper/test-other/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

In [ ]:
src = Path(SOURCE)
dest = Path(DESTINATION)
dest.mkdir(parents=True, exist_ok=True)

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

    print(f"Processing")
    print(f"\tAudio name: {flac.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")
    start_time = time.perf_counter()
    segments, _= model.transcribe(audio, beam_size=5)
    trn = segments_to_sclite_trn(flac.stem, segments)

    end_time = time.perf_counter()
    print(f"Processed time: {end_time - start_time:.2f} seconds")
    trn.text = normalize_text_only_en(trn.text).upper()
    return trn

In [ ]:
%%time
make_all_ref_and_hyp(src, dest, transcriber, 1)

In [ ]:
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.ref.trn"))),
    dest / "concat.ref.trn"
)
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.hyp.trn"))),
    dest / "concat.hyp.trn"
)

In [ ]:
output = sclite_trn_run(
    dest / "concat.ref.trn",
    dest / "concat.hyp.trn",
)

In [ ]:
parse_sclite_summary(output)

# {'num_sentences': 96,
#  'num_words': 1472,
#  'correct_percent': 93.3,
#  'substitution_percent': 6.2,
#  'deletion_percent': 0.5,
#  'insertion_percent': 0.9,
#  'wer_percent': 7.5,
#  'sentence_error_percent': 56.3}